# Module 4 · Filtering the Dataset (with AI, checked)

*Jupyter Book, notebook 2 of 3: M2 obtain > **M4 filter** > M5 analyse.*

In Module 2 we saw the combined Van Gogh dataset is dominated by one **reproduction index**. Here we make the data **fit for analysis** by filtering, and we do it the course's way: draft the code with AI, then **check and document** it.

**No API key needed**, this notebook works on the file Module 2 produced.


## 1 · Load the dataset from Module 2


In [1]:
from pathlib import Path
import pandas as pd

# Candidate locations for the shared datasets/ folder (repo root, or one level up from a module).
_DATASET_NAMES = ["van_gogh_combined.csv", "van_gogh_europeana.csv", "van_gogh_filtered.csv"]

def find_dataset(name):
    """Return the path to a dataset file, wherever the notebook is run from."""
    for base in [Path("datasets"), Path("../datasets")]:
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(f"{name} not found in datasets/, run the earlier notebook first.")

def datasets_dir():
    """The datasets/ folder that actually holds the data (so saves land beside the real files)."""
    for base in [Path("datasets"), Path("../datasets")]:
        if any((base / n).exists() for n in _DATASET_NAMES):
            return base
    Path("datasets").mkdir(parents=True, exist_ok=True)   # first run, live mode
    return Path("datasets")

def load_combined():
    for name in ["van_gogh_combined.csv", "van_gogh_europeana.csv"]:
        try:
            p = find_dataset(name); print("Loaded", p); return pd.read_csv(p)
        except FileNotFoundError:
            continue
    raise FileNotFoundError("Run Module 2 first, or check datasets/.")

df = load_combined()
print("Records:", len(df))
df["institution"].value_counts().head(8)

Loaded ../datasets/van_gogh_combined.csv
Records: 334


institution
German Documentation Center for Art History - Marburg Picture Index    300
Catholic University of Leuven                                           18
International Institute of Social History                                4
Digital Library for Dutch Literature                                     4
Austrian Gallery Belvedere                                               2
The Israel Museum, Jerusalem                                             1
Nationalmuseum Sweden                                                    1
Wellcome Collection                                                      1
Name: count, dtype: int64

> **Why the record count may jump here.** If you ran Module 2 **without an API key**, it used the
> small 24-record sample. This module loads `van_gogh_combined.csv`, the **full result of the live,
> multi-variant search** (334 records), which ships with the course so you can continue without a key.
> With a key, Module 2 regenerates that same combined file. Either way, the *method* is what matters:
> combine name variants, then filter.


## 2 · Decide the filter, and say why

Our research question is about **institutions that hold Van Gogh's works**. One “institution” in the data is a **photographic reproduction index** (the Marburg Picture Index), not a holding museum, so for *this* question we set it aside. This is a **documented decision**, not silent deletion: we keep a copy and record what we did.

> **Prompt you could use (then check it!):** *In pandas, from `df`, create `filtered` that removes rows whose `institution` is in a list of reproduction/aggregator providers I give you. Do not modify the original. Print how many rows were removed and how many remain.*
>
> **Always check AI code.** Read it, run it on a small piece, and confirm the row counts before and after make sense. The AI can pick the wrong column or drop too much.


In [2]:
# Providers that are reproduction/aggregation indexes rather than holding institutions.
# (Add to this list for your own dataset, this is the adjustable part.)
REPRODUCTION_PROVIDERS = [
    "German Documentation Center for Art History - Marburg Picture Index",
]

filtered = df[~df["institution"].isin(REPRODUCTION_PROVIDERS)].copy()   # ~ means "not in"
print(f"Removed {len(df) - len(filtered)} reproduction-index rows; {len(filtered)} remain.")
filtered["institution"].value_counts()

Removed 300 reproduction-index rows; 34 remain.


institution
Catholic University of Leuven                18
International Institute of Social History     4
Digital Library for Dutch Literature          4
Austrian Gallery Belvedere                    2
The Israel Museum, Jerusalem                  1
Nationalmuseum Sweden                         1
Wellcome Collection                           1
Digital Memory of Catalonia                   1
Finnish National Gallery                      1
Swedish Air Force Museum                      1
Name: count, dtype: int64

## 3 · A second, gentler filter (optional, adjustable)

You can also keep only records useful for your question, e.g. those with a known `year`, or a given `type`. Here we simply drop rows with no institution. Change these lines to suit your own analysis.


In [3]:
filtered = filtered[filtered["institution"].notna() & (filtered["institution"].astype(str) != "")].copy()
print("After tidying:", len(filtered), "records across", filtered["institution"].nunique(), "institutions")

After tidying: 34 records across 10 institutions


## 4 · Save the filtered dataset for Module 5


In [4]:
out = datasets_dir() / "van_gogh_filtered.csv"
filtered.to_csv(out, index=False)
print("Saved ->", out)

Saved -> ../datasets/van_gogh_filtered.csv


**Workflow note (for your README):** *Filtered the combined dataset for the institution question by setting aside the Marburg reproduction index (documented decision, original kept) and dropping rows with no institution; result saved as `van_gogh_filtered.csv`.*

**Next: Module 5, Analysis & visualisation.**
